# Demo: Build a Machine Learning Model in Amazon SageMaker AI Notebooks

**Level:** Beginner  
**Estimated time:** 25–35 minutes  
**Use case:** Predict an iris flower species from four flower measurements  
**Technique:** Supervised learning — classification with Logistic Regression

## Learning objectives
By the end of this demo, you will be able to:
1. Load and inspect a dataset.
2. Separate features from the target label.
3. Split data into training and testing sets.
4. Train a machine learning model.
5. Evaluate model performance and make a prediction.
6. Save and reload a trained model.

> **Scope:** This demo trains the model directly on the notebook's compute using scikit-learn. It does **not** create a SageMaker managed training job or a hosted endpoint. This keeps the first classroom demo simple and avoids endpoint charges.

## Step 1 — Check the environment

Open this notebook in Amazon SageMaker AI Notebooks / Studio, select a Python kernel, and run each cell from top to bottom. The common data-science libraries used here are typically available in SageMaker AI notebook environments.

In [ ]:
import sys
import sklearn
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)

## Step 2 — Load the sample dataset

The Iris dataset is bundled with scikit-learn, so this demo does not need a separate download or an S3 bucket. Each row describes a flower using four measurements. The target is the flower species.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()
feature_names = iris.feature_names
target_names = iris.target_names

print("Dataset rows and columns:", df.shape)
display(df.head())
print("Species labels:", list(target_names))

## Step 3 — Understand features and target

- **Features (X):** sepal length, sepal width, petal length, and petal width.
- **Target (y):** the species encoded as `0`, `1`, or `2`.

In real projects, features are the information the model uses to predict the target.

In [ ]:
X = df[feature_names]
y = df["target"]

print("Feature columns:", list(X.columns))
print("Target counts:")
print(y.map(dict(enumerate(target_names))).value_counts())

## Step 4 — Split into training and testing data

We train on 80% of the examples and hold back 20% for evaluation. `stratify=y` helps preserve the species proportions in both sets. `random_state=42` makes the split repeatable for teaching.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Step 5 — Build and train the model

Logistic Regression is a commonly used classification algorithm. This pipeline first standardizes the input features, then trains the classifier. Calling `fit()` is the training step.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42)
)

model.fit(X_train, y_train)
print("Model training complete.")

## Step 6 — Evaluate the model

Accuracy is the fraction of test examples classified correctly. The classification report also shows precision, recall, and F1-score for each species. Evaluation uses the held-back test data, which the model did not use during fitting.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.2%}")
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=target_names))

### View a confusion matrix

Rows are the true species and columns are the predicted species. Correct predictions appear on the main diagonal.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=target_names,
    cmap="Blues"
)
plt.title("Iris Species — Confusion Matrix")
plt.tight_layout()
plt.show()

## Step 7 — Predict a new flower

Enter four measurements in the same order as the feature columns: sepal length, sepal width, petal length, and petal width (all in centimetres). The example below uses sample measurements resembling an iris flower.

In [ ]:
new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=feature_names
)

predicted_class = int(model.predict(new_flower)[0])
predicted_species = target_names[predicted_class]
probabilities = model.predict_proba(new_flower)[0]

print("Input measurements:")
display(new_flower)
print("Predicted species:", predicted_species)
print("Class probabilities:")
for species, probability in zip(target_names, probabilities):
    print(f"  {species}: {probability:.2%}")

## Step 8 — Save and reload the trained model

Saving a model lets you reuse it later without retraining. This writes a file into the notebook's current working directory.

In [ ]:
import joblib
from pathlib import Path

model_path = Path("iris_classifier.joblib")
joblib.dump(model, model_path)
print("Saved model to:", model_path.resolve())

loaded_model = joblib.load(model_path)
reloaded_prediction = int(loaded_model.predict(new_flower)[0])
print("Prediction from reloaded model:", target_names[reloaded_prediction])

## Step 9 — Student practice

Try one change at a time and rerun the affected cells:

1. Change `test_size=0.20` to `test_size=0.30`. Does test accuracy change?
2. Change the measurements in `new_flower`. Does the predicted species change?
3. Replace `LogisticRegression` with `RandomForestClassifier` from `sklearn.ensemble`. Compare test accuracy.
4. Explain why we evaluate on test data instead of only reporting training accuracy.

## Key takeaway

The basic ML workflow is **load data → select features and target → split data → train → evaluate → predict → save the model**. SageMaker AI Notebooks provide the managed notebook environment; this introductory example uses scikit-learn locally within that environment.